In [ ]:
import yfinance as yf
from fredapi import Fred
import pandas as pd
import pandas_datareader.data as web
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import numpy as np
import os

# Helper Functions

In [ ]:

# ********************************************************************************************************
# Helper Functions
# ********************************************************************************************************

def run_strategy(df, initial_capital, leverage):

    df_strat = df.copy()
    n = len(df_strat)

    theta = np.zeros(n)  # Dollar position in the risky asset

    V = np.zeros(n) # Risky asset portfolio portion
    dV = np.zeros(n) # Daily PnL from risky asset

    V_cap = np.zeros(n)  # Cash portion (money-market)
    dV_cap = np.zeros(n)  # Daily PnL from cash

    V_tot = np.zeros(n)  # Total portfolio value
    dV_tot = np.zeros(n)  # Total daily PnL

    # ---- Initialization ----
    V_tot[0] = initial_capital      # Total starting capital
    V_cap[0] = initial_capital     # Initially, all capital is cash (no risky exposure)
    V[0] = 0                # No allocation in the risky asset at the start

    theta[0] = df_strat['final_signal'].iloc[0]*V_tot[0]*leverage


    for t in range(1, n):
        risky_position = theta[t-1] # Dollar-Value of the position in the risky asset

        # Daily PnL of Risky asset  & Update of its value
        dV[t] = df_strat['excess_returns'].iloc[t]*theta[t-1]
        V[t] = V[t-1] + dV[t]

        # Daily PnL of the Cash Position & Update of its value
        M_t = abs(risky_position)/leverage
        dV_cap[t] = (V_tot[t-1] - M_t)*df_strat['rf_daily'].iloc[t] 
        V_cap[t] = V_cap[t-1] + dV_cap[t]

        # Total Daily PnL & Update of the total portfolio value
        dV_tot[t] = dV[t] + dV_cap[t]
        V_tot[t] = V_cap[t] + V[t]

        # Update position in risky asset for the next trading day
        theta[t] = df_strat['final_signal'].iloc[t]*V_tot[t]*leverage

    strategy_returns = {
        'V': V,
        'dV': dV,
        'V_cap': V_cap,
        'dV_cap': dV_cap,
        'V_tot': V_tot,
        'dV_tot': dV_tot,
        'theta': theta
    }

    return strategy_returns



# PERFORMANCE METRICS

def compute_performance_metrics(strategy_dict):
    # Daily Returns
    total_returns = strategy_dict['dV_tot'][1:]/strategy_dict['V_tot'][:-1]

    # Sharpe Ratio
    expected_total_return = np.mean(total_returns)
    vol_total_return = np.std(total_returns)

    sharpe_ratio = (expected_total_return * 252) / (vol_total_return * np.sqrt(252)) if vol_total_return > 0 else np.nan


    # Maximum Drawdown
    daily_total_pnl = strategy_dict['dV_tot'][1:]
    cum_total_pnl = np.cumsum(daily_total_pnl)

    peak_value = np.maximum.accumulate(cum_total_pnl)
    # print(peak_value)
    drawdown = (cum_total_pnl - peak_value) / peak_value

    max_drawdown = np.min(drawdown)

    # Calmar Ratio
    if max_drawdown < 0:
        calmar_ratio = - 1 * (expected_total_return * 252) / max_drawdown
    else: 
        calmar_ratio = np.nan

    performance_metrics = {'Sharpe': sharpe_ratio,
            'Max Drawdown': max_drawdown,
            'Calmar Ratio': calmar_ratio,
            'Total Return': cum_total_pnl[-1]}

    return performance_metrics


def plot_in_out_daily_pnl(in_results, out_results, title_prefix="Strategy"):
    
    # time axes
    n_ins = len(in_results['dV_tot'])
    t_ins = np.arange(n_ins)
    t_oos = n_ins + np.arange(len(out_results['dV_tot']))

    fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=False)
    
    # In-Sample Daily PNLs
    ax[0].plot(t_ins, in_results['dV'],     label='Risky Asset - Daily PnL', color='tab:blue', lw=0.8)
    ax[0].plot(t_ins, in_results['dV_cap'], label='Cash - Daily PnL', color='tab:green', lw=0.8)
    ax[0].plot(t_ins, in_results['dV_tot'], label='Total Daily PnL', color='tab:red', 
               lw=0.8, marker='o', markersize=5, markevery=[0, -1], markerfacecolor='tab:red',
               markeredgecolor='none')
    ax[0].set_title("In-Sample")
    ax[0].set_ylabel("PnL ($)")
    ax[0].legend(loc='upper right')
    ax[0].grid(alpha=0.3)

    # Out of Sample Daily PNLs
    ax[1].plot(t_oos, out_results['dV'],     label='Risky Asset - Daily PnL', color='tab:blue', lw=0.8)
    ax[1].plot(t_oos, out_results['dV_cap'], label='Cash - Daily PnL', color='tab:green', lw=0.8)
    ax[1].plot(t_oos, out_results['dV_tot'], label='Total Daily PnL', color='tab:red', lw=0.8,
               marker='o', markersize=5, markevery=[0, -1], markerfacecolor='tab:red', markeredgecolor='none')
    ax[1].set_title("Out-of-Sample")
    ax[1].set_xlabel("Time (days)")
    ax[1].set_ylabel("PnL ($)")
    ax[1].grid(alpha=0.3)

    for a in ax:
        fmt = ScalarFormatter(useOffset=False)
        fmt.set_scientific(False)
        a.yaxis.set_major_formatter(fmt)

    plt.tight_layout()
    plt.show()


def plot_in_out_cum_pnl(in_results, out_results, title_prefix="Strategy"):

    # time axes
    n_ins = len(in_results['dV_tot'])
    t_ins = np.arange(n_ins)
    t_oos = n_ins + np.arange(len(out_results['dV_tot']))

    # cumulative sums
    cum_ins_risky = np.cumsum(in_results['dV'])
    cum_ins_cash  = np.cumsum(in_results['dV_cap'])
    cum_ins_tot   = np.cumsum(in_results['dV_tot'])

    cum_oos_risky = np.cumsum(out_results['dV'])
    cum_oos_cash  = np.cumsum(out_results['dV_cap'])
    cum_oos_tot   = np.cumsum(out_results['dV_tot'])

    # create figure + axes
    fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=False)

    # In‑sample cumulative PnL
    ax[0].plot(t_ins, cum_ins_risky, label='Risky Asset - Cum PnL', color='tab:blue', lw=0.8)
    ax[0].plot(t_ins, cum_ins_cash,  label='Cash - Cum PnL',  color='tab:green', lw=0.8)
    ax[0].plot(t_ins, cum_ins_tot,   label='Total Cum PnL', color='tab:red', lw=0.8,
               marker='o',markersize=5, markevery=[0, -1], markerfacecolor='tab:red', markeredgecolor='none')
    ax[0].set_title("In-Sample")
    ax[0].set_ylabel("Cumulative PnL ($)")
    ax[0].legend(loc='upper left')
    ax[0].grid(alpha=0.3)

    # Out‑of‑sample cumulative PnL
    ax[1].plot(t_oos, cum_oos_risky, label='Risky', color='tab:blue', lw=0.8)
    ax[1].plot(t_oos, cum_oos_cash,  label='Cash',  color='tab:green', lw=0.8)
    ax[1].plot(t_oos, cum_oos_tot,   label='Total', color='tab:red', lw=0.8,
               marker='o', markersize=5, markevery=[0, -1], markerfacecolor='tab:red', markeredgecolor='none')
    ax[1].set_title("Out-of-Sample")
    ax[1].set_xlabel("Time (Days)")
    ax[1].set_ylabel("Cumulative PnL ($)")
    ax[1].grid(alpha=0.3)

    for a in ax:
        fmt = ScalarFormatter(useOffset=False)
        fmt.set_scientific(False)
        a.yaxis.set_major_formatter(fmt)

    plt.tight_layout()
    plt.show()

# Data Retrieval
Replace "your_fred_api_key_here" with your actual fred api key.

In [ ]:
# ********************************************************************************************************
# Data Preprocessing 
# ********************************************************************************************************

# Time Period
start_date = "2023-01-01"
end_date = "2023-12-31"
ticker = "SPTL"

# SPTL-ETF DATA
sptl = yf.download(ticker, start=start_date, end=end_date)
sptl.index = pd.to_datetime(sptl.index, format='%Y-%m-%d')
sptl.columns = sptl.columns.droplevel(1)
sptl.rename_axis(None, axis=1, inplace=True)
sptl.drop(columns=["Open"], inplace=True)

# EFFR DATA
fred = Fred(api_key="your_fred_api_key_here")  # Replace with your actual FRED API key
effr_series = fred.get_series("EFFR", observation_start=start_date, observation_end=end_date)
effr_data = pd.DataFrame(effr_series, columns=["EFFR"])
effr_data["EFFR"] = (effr_data["EFFR"] / 100.0)
effr_data.index = pd.to_datetime(effr_data.index, format='%Y-%d-%m')
effr_data["EFFR"] = effr_data["EFFR"].ffill()

# Merging SPTL-ETF and EFFR Data
merged_df = pd.merge(sptl, effr_data, left_index=True, right_index=True, how='left')

# Creating Daily EFFR Rate - Taken as the Daily Risk-free Rate
merged_df['rf_daily'] = merged_df['EFFR'] / 252.0

# Daily Returns & Excess SPTL Returns
merged_df['SPTL_returns'] = merged_df['Close'].pct_change()
merged_df['excess_returns'] = merged_df['SPTL_returns'] - merged_df['rf_daily']

# Removing first row (NaN value for return calculations)
merged_df.dropna(inplace=True)
merged_df.head()

# Signal Generation Helper Functions
Mean Reversion Signal: The signal is scaled by the size of the deviation between the moving average and the current return, and clipped between -1 and 1.

Baseline Signal: Buy and Hold.

In [ ]:
# ********************************************************************************************************
# Signal Generation
# ********************************************************************************************************
def mean_reversion_signal(df, window):

    ret = df.copy()

    # 1) Calculating rollinge expected return and volatility
    ret['expected_return'] = ret['SPTL_returns'].rolling(window=window).mean()
    ret['volatility'] = ret['SPTL_returns'].rolling(window=window).std()

    # 2) a) Computing the deviation (raw signal) deviation
    ret['deviation'] = ret['SPTL_returns'] - ret['expected_return']

    # 3) Generating the CONTINUOUS thresholded signal
    ret['signal'] =  -ret['deviation']
    ret['signal'] = ret['signal'].clip(lower=-1, upper=1) # Bounding the signal to [-1, 1]

    ret['final_signal'] = ret['signal'].shift(1)

    return ret.dropna()

In [ ]:
# BASELINE SIGNAL

def buy_and_hold_signal(df):
    df_signal = df.copy()
    df_signal['final_signal'] = 1

    return df_signal

# Model Selection
Initial fit on training data and hyperparameter (moving average lookback window size) tuning on validation set (validation sharpe).

In [ ]:
# MODEL SELECTION --> INITIAL TRAINING FIT + HYPERPARAMETER TUNING ON VALIDATION SET


df = merged_df.copy()
N = len(df)
train_end = int(N * 0.50)  
val_end   = int(N * 0.80)    
train_df  = df.iloc[:train_end]
val_df    = df.iloc[train_end:val_end]
test_df   = df.iloc[val_end:]

# 2) grid‐search on TRAIN but evaluate on VAL
best_window = None
best_val_sharpe = -np.inf
results = {}
windows = [8, 10, 12, 15, 20]
rows = []

for w in windows:
    
        # Fit on the training sample (50%)
        train_sig = mean_reversion_signal(train_df, window=w)
        train_res = run_strategy(train_sig, initial_capital=100000.0, leverage=10.0)
        train_metrics = compute_performance_metrics(train_res)
        
        # Evaluate Sharpe on training sample (next 30%)
        val_sig = mean_reversion_signal(val_df, window=w)
        val_res = run_strategy(val_sig, initial_capital=100000.0, leverage=10.0)
        val_metrics = compute_performance_metrics(val_res)

        rows.append({
            'window': w,
            'train Sharpe': train_metrics['Sharpe'],
            'val Sharpe':   val_metrics['Sharpe']
        })


        val_sharpe = val_metrics['Sharpe']
        results[w] = val_metrics
    
        if val_sharpe > best_val_sharpe:
            best_val_sharpe = val_sharpe
            best_window = w

table = pd.DataFrame(rows).set_index('window')
table = table.sort_values('val Sharpe', ascending=False)
print(f"→ Best window by VAL Sharpe: {best_window} (Val Sharpe={best_val_sharpe:.2f})")
display(table)


# Final model run on:
1) Full in-sample data (full training + validation set)
2) Full out-of-sample data (held out and completely unseen test set)

Generation of performance metrics and plotsfor both runs (in & out-of-sample data)

In [ ]:
# FINAL MODEL RUN

# ********************************************************************************************************
# Evaluate Final Model on FULL IN_SAMPLE & FULL OUT_Sample
# ********************************************************************************************************
full_in_sample = pd.concat([train_df, val_df])
in_results = run_strategy(mean_reversion_signal(full_in_sample, window=best_window),
                           initial_capital=100000.0,
                           leverage=10.0)
in_metrics = compute_performance_metrics(in_results)


full_out_sample = test_df.copy()
oos_results = run_strategy(mean_reversion_signal(full_out_sample, window=best_window),
                           initial_capital=100000.0,
                           leverage=10.0)

oos_metrics = compute_performance_metrics(oos_results)

summary = pd.DataFrame({
    'Sharpe Ratio':   [in_metrics['Sharpe'],   oos_metrics['Sharpe']],
    'Max Drawdown':   [in_metrics['Max Drawdown'], oos_metrics['Max Drawdown']],
    'Calmar Ratio':   [in_metrics['Calmar Ratio'], oos_metrics['Calmar Ratio']],
    'Total Return':   [in_metrics['Total Return'], oos_metrics['Total Return']],
}, index=['In‑Sample','Out‑of‑Sample']).round(3)
display(summary)


In [ ]:
# PLOTTING FINAL IN-SAMPLE vs OUT-SAMPLE Daily PnL Performance
plot_in_out_daily_pnl(in_results,  oos_results, title_prefix="Mean Reversion Strategy PnL")

In [ ]:
# PLOTTING FINAL IN-SAMPLE vs OUT-SAMPLE Cummulative PnL Performance
plot_in_out_cum_pnl (in_results,  oos_results, title_prefix="Mean Reversion Strategy PnL")